In [7]:
import pandas as pd
from transformers import pipeline

In [4]:
df = pd.read_csv("../data/raw/bank_reviews_cleaned.csv")

In [5]:
print(df.head())

                    review  rating        date bank       source
0  it's a good application       5  2026-05-13  CBE  Google Play
1            thank you cbe       5  2026-05-13  CBE  Google Play
2                  is good       5  2026-05-13  CBE  Google Play
3                      wow       5  2026-05-13  CBE  Google Play
4         Good application       2  2026-05-13  CBE  Google Play


In [6]:
print(df.shape)

(1500, 5)


In [8]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\Dataencoder\fintech-review-analytics\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dataencoder\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [9]:
test_review = "The app is fast and very easy to use."

result = sentiment_pipeline(test_review)

print(result)

[{'label': 'POSITIVE', 'score': 0.998571515083313}]


In [10]:
bad_review = "Transfers fail constantly and login is broken."

result = sentiment_pipeline(bad_review)

print(result)

[{'label': 'NEGATIVE', 'score': 0.9996556043624878}]


In [11]:
def get_sentiment(text):

    try:
        result = sentiment_pipeline(text[:512])[0]

        label = result['label']
        score = result['score']

        return pd.Series([label, score])

    except:
        return pd.Series(["UNKNOWN", 0])

In [12]:
df[['sentiment_label', 'sentiment_score']] = df['review'].apply(get_sentiment)

In [13]:
print(df[['review', 'sentiment_label', 'sentiment_score']].head())

                    review sentiment_label  sentiment_score
0  it's a good application        POSITIVE         0.999866
1            thank you cbe        POSITIVE         0.999756
2                  is good        POSITIVE         0.999839
3                      wow        POSITIVE         0.999592
4         Good application        POSITIVE         0.999855


In [14]:
def add_neutral(label, score):

    if score < 0.60:
        return "NEUTRAL"

    return label

In [15]:
df['sentiment_label'] = df.apply(
    lambda row: add_neutral(
        row['sentiment_label'],
        row['sentiment_score']
    ),
    axis=1
)

In [16]:
print(df['sentiment_label'].value_counts())

sentiment_label
POSITIVE    914
NEGATIVE    576
NEUTRAL      10
Name: count, dtype: int64


In [17]:
df.to_csv("../data/raw/sentiment_reviews.csv", index=False)